# พรีวิวก่อนเทรน — ดูว่าเส้นฐานอยู่ตรงไหน

โน้ตบุ๊กนี้ **ไม่เทรนอะไรเลย** ใช้ SDXL เปล่า ๆ เจนด้วย prompt ชุดเดียวกับที่จะใช้
ทดสอบหลังเทรน เพื่อให้เห็นว่า

- ภาพที่ออกมาหน้าตาเป็นยังไง ขนาดเท่าไร ใช้งานได้จริงไหม
- SDXL เปล่าทำอะไรได้แค่ไหนเมื่อสั่งว่าอยากได้ผ้ามัดหมี่
- ช่องว่างที่ LoRA ต้องเติมคืออะไร

ใช้เวลาราว **10 นาที** เทียบกับเทรนจริง 1 ชั่วโมง

**ก่อนเริ่ม** Runtime → Change runtime type → **T4 GPU**


## 1. ตรวจ GPU และติดตั้ง


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Colab ติดตั้ง torchao เก่ามาให้ ซึ่งชนกับ peft เอาออกก่อน
!pip uninstall -y -q torchao

# --no-deps สำคัญ ห้ามให้ pip แตะ torch ของ Colab
# ถ้าปล่อยให้อัปเกรดตามใจ มันจะลง torch รุ่น CPU ทับแล้ว GPU หายทั้งรันไทม์
!pip install -q --no-deps -U diffusers peft

import torch, diffusers
print('torch', torch.__version__, '| CUDA ใช้ได้:', torch.cuda.is_available())
print('diffusers', diffusers.__version__)
assert torch.cuda.is_available(), (
    'torch ไม่มี CUDA ให้ Runtime > Disconnect and delete runtime แล้วเริ่มใหม่'
)
print('GPU:', torch.cuda.get_device_name(0))


## 2. โหลด SDXL

ดาวน์โหลดราว 7 GB ครั้งแรกใช้เวลา 2-4 นาที


In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
).to('cuda')
pipe.set_progress_bar_config(disable=True)
print('พร้อมเจนแล้ว')


## 3. เจนด้วย prompt ชุดทดสอบ

prompt ชุดนี้เหมือนกับที่จะใช้หลังเทรนทุกตัวอักษร
ตอนนี้ยังไม่มี LoRA คำว่า `isaanmudmee` จึงไม่มีความหมายกับโมเดล
สิ่งที่เห็นคือ SDXL ตีความคำบรรยายล้วน ๆ


In [ ]:
TRIGGER = 'isaanmudmee'

PROMPTS = {
  'A': TRIGGER + ', naga motif with flame crest, burnt orange ground, cream figures',
  'B': TRIGGER + ', tiered offering motif with finial, violet ground, white figures',
  'C': TRIGGER + ', concentric lozenge lattice with floral centre, indigo ground',
  'D': TRIGGER + ', S-hook motif between warp bands, dark maroon ground, cream figures',
  'E': TRIGGER + ', naga motif inside lozenge lattice, indigo ground, three tones of blue',
  'F': TRIGGER + ', tiered offering motif with hook filler, green ground, gold figures',
  'G': TRIGGER + ', spiral and scroll motif, two tone gold and black',
  'H': TRIGGER + ', lotus motif, pink ground, white figures',
  'I': TRIGGER + ', horizontal banded repeat, teal ground',
  'J': TRIGGER,
}
NEGATIVE = 'photograph of a person, text, watermark, blurry, frame, border, 3d render'
SEED = 1337

import pathlib, time
out = pathlib.Path('/content/preview'); out.mkdir(exist_ok=True)

started = time.time()
for key, prompt in PROMPTS.items():
    generator = torch.Generator('cuda').manual_seed(SEED)
    image = pipe(
        prompt=prompt, negative_prompt=NEGATIVE,
        num_inference_steps=28, guidance_scale=6.0,
        width=1024, height=1024, generator=generator,
    ).images[0]
    image.save(out / (key + '.png'))
    print(key, 'เสร็จ', round(time.time() - started), 'วินาที')

print('ทั้งหมด', round(time.time() - started), 'วินาที')


## 4. ดูผล


## 5. เลือกผ้าต้นแบบเอง

เหมือนกดปุ่ม + บนการ์ดในเว็บ เลือกกี่ผืนก็ได้
คำบรรยายลายของแต่ละผืนมาจาก `manifest.json` ชุดเดียวกับที่ใช้เทรน
prompt จึงถูกประกอบแบบเดียวกับที่เว็บจริงทำ


In [ ]:
PATTERNS = {
  'nong-khai':         {'thai': 'ลายนาคใหญ่ (จ.หนองคาย)', 'family': 'naga',
                        'motif': 'naga motif with flame crest',
                        'composition': 'vertical axis repeat', 'colors': 'burnt orange ground, cream and pale grey figures'},
  'kalasin':           {'thai': 'ลายนาค 12 แขน (จ.กาฬสินธุ์)', 'family': 'naga',
                        'motif': 'many-armed naga motif',
                        'composition': 'nested lozenge frames', 'colors': 'dark plum ground, cream and gold figures'},
  'bueng-kan':         {'thai': 'ลายหมากเบ็ง (จ.บึงกาฬ)', 'family': 'phum',
                        'motif': 'tiered offering motif with finial',
                        'composition': 'single row along the lower edge', 'colors': 'bright violet ground, white figures'},
  'amnat-charoen':     {'thai': 'ลายตะขอสลับเอื้อ (จ.อำนาจเจริญ)', 'family': 'kho',
                        'motif': 'hook and interlock motif',
                        'composition': 'dense all-over lattice', 'colors': 'deep indigo ground, lighter blue figures'},
  'chaiyaphum':        {'thai': 'ลายหมี่คั่นขอนารี (จ.ชัยภูมิ)', 'family': 'kho',
                        'motif': 'S-hook motif between warp bands',
                        'composition': 'vertical stripe bands', 'colors': 'dark maroon ground, cream and rust figures'},
  'sakon-nakhon':      {'thai': 'ลายนครธรรม (จ.สกลนคร)', 'family': 'lozenge',
                        'motif': 'concentric lozenge lattice with floral centre',
                        'composition': 'half-drop all-over repeat', 'colors': 'indigo ground, three tones of blue and white'},
  'si-sa-ket':         {'thai': 'ลายลูกแก้ว (จ.ศรีสะเกษ)', 'family': 'lozenge',
                        'motif': 'small raised lozenge motif',
                        'composition': 'fine all-over repeat', 'colors': 'golden ochre monochrome'},
  'nakhon-phanom':     {'thai': 'ลายผ้ามุก (จ.นครพนม)', 'family': 'lozenge',
                        'motif': 'cross and lozenge motif',
                        'composition': 'vertical stripe bands', 'colors': 'dark purple ground, white and red figures'},
  'yasothon':          {'thai': 'ลายยศสุนทร (จ.ยโสธร)', 'family': 'lozenge',
                        'motif': 'interlaced rattan lattice',
                        'composition': 'dense all-over field with banded border', 'colors': 'bright cyan ground, white figures'},
  'mukdahan':          {'thai': 'ลายแก้วมุกดา (จ.มุกดาหาร)', 'family': 'lozenge',
                        'motif': 'faceted gem motif with small naga',
                        'composition': 'horizontal banded repeat', 'colors': 'strong blue ground, gold and pink accents'},
  'loei':              {'thai': 'ลายดอกฝ้ายเมืองเลย (จ.เลย)', 'family': 'floral',
                        'motif': 'cotton flower motif',
                        'composition': 'vertical stripe bands', 'colors': 'indigo ground, white figures'},
  'maha-sarakham':     {'thai': 'ลายสร้อยดอกหมาก (จ.มหาสารคาม)', 'family': 'floral',
                        'motif': 'hanging flower chain motif',
                        'composition': 'horizontal banded repeat', 'colors': 'dark brown ground, green yellow and red accents'},
  'nong-bua-lamphu':   {'thai': 'ลายขิดสลับหมี่ (จ.หนองบัวลำภู)', 'family': 'floral',
                        'motif': 'lotus motif with raised supplementary weft',
                        'composition': 'vertical stripe bands', 'colors': 'magenta pink ground, white figures'},
  'ubon-ratchathani':  {'thai': 'ลายกาบบัว (จ.อุบลราชธานี)', 'family': 'floral',
                        'motif': 'lotus sheath motif',
                        'composition': 'vertical stripe bands', 'colors': 'orchid purple ground, white and pale pink figures'},
  'udon-thani':        {'thai': 'ลายก้นหอย (จ.อุดรธานี)', 'family': 'spiral',
                        'motif': 'spiral and scroll motif',
                        'composition': 'horizontal banded repeat', 'colors': 'burnt orange ground, cream figures'},
  'surin':             {'thai': 'ลายโฮล (จ.สุรินทร์)', 'family': 'stripe',
                        'motif': 'weft-tied chevron motif',
                        'composition': 'vertical ikat columns', 'colors': 'dusty rose and grey green'},
  'buri-ram':          {'thai': 'ลายหางกระรอกคู่ (จ.บุรีรัมย์)', 'family': 'stripe',
                        'motif': 'twisted two-tone yarn texture',
                        'composition': 'fine horizontal striation', 'colors': 'mauve and smoke grey iridescent'},
  'roi-et':            {'thai': 'ลายสาเกต (จ.ร้อยเอ็ด)', 'family': 'stripe',
                        'motif': 'banded supplementary weft motif',
                        'composition': 'horizontal banded repeat', 'colors': 'hot pink ground, gold and white accents'},
}

print('ลายผ้าที่เลือกได้', len(PATTERNS), 'ผืน')
for key, value in PATTERNS.items():
    print(' ', key.ljust(20), value['thai'])


แก้ `เลือก` เป็นรหัสผ้าที่อยากผสม แล้วกดรัน

ภาพผ้าจริงจะถูกดึงมาแสดงข้างซ้าย เทียบกับภาพที่โมเดลเจนทางขวา
จะได้เห็นชัดว่าตอนนี้ห่างกันแค่ไหน และ LoRA ต้องเติมอะไร


In [ ]:
SELECT = ['bueng-kan', 'chaiyaphum']   # ผ้าที่อยากผสม ใส่รหัสจากรายการข้างบน
EXTRA  = ''                            # คำขอเพิ่มเติม เช่น 'tiny stars scattered above'

import random, urllib.request, io as _io
from PIL import Image
import matplotlib.pyplot as plt

picked = [PATTERNS[k] for k in SELECT if k in PATTERNS]
assert picked, 'ไม่พบรหัสผ้าที่ใส่มา ดูรายการจากเซลล์ข้างบน'

# ประกอบ prompt แบบเดียวกับที่เว็บทำ: ลายหลักจากผืนแรก
# แล้วเอาลายของผืนที่เหลือมาเป็นลายเติมช่องว่าง
head = picked[0]
parts = [TRIGGER, head['motif']]
for other in picked[1:]:
    short = other['motif'].split(' motif')[0]
    parts.append('with ' + ' '.join(short.split()[:3]) + ' fillers')
parts.append(head['composition'])
parts.append(head['colors'])
if EXTRA.strip():
    parts.append(EXTRA.strip())
prompt = ', '.join(parts)
print(prompt)

seed = random.randrange(2**31)
generator = torch.Generator('cuda').manual_seed(seed)
image = pipe(
    prompt=prompt, negative_prompt=NEGATIVE,
    num_inference_steps=28, guidance_scale=6.0,
    width=1024, height=1024, generator=generator,
).images[0]

# ดึงรูปผ้าจริงจาก GitHub มาเทียบ
RAW = 'https://raw.githubusercontent.com/BallKKu/Matin/main/public/silk/'
reference = []
for key in SELECT:
    try:
        with urllib.request.urlopen(RAW + key + '.jpg', timeout=20) as response:
            reference.append((key, Image.open(_io.BytesIO(response.read()))))
    except Exception as error:
        print('ดึงรูป', key, 'ไม่ได้:', error)

columns = len(reference) + 1
fig, axes = plt.subplots(1, columns, figsize=(6 * columns, 6))
if columns == 1:
    axes = [axes]
for ax, (key, img) in zip(axes, reference):
    ax.imshow(img); ax.set_title('ผ้าจริง · ' + key, fontsize=11); ax.axis('off')
axes[-1].imshow(image)
axes[-1].set_title('โมเดลเจน (ยังไม่มี LoRA) · seed ' + str(seed), fontsize=11)
axes[-1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
import pathlib
from PIL import Image
import matplotlib.pyplot as plt

files = sorted(pathlib.Path('/content/preview').glob('*.png'))
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, path in zip(axes.flat, files):
    ax.imshow(Image.open(path))
    ax.set_title(path.stem + ' — ' + PROMPTS[path.stem][12:60], fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()


## 6. ลองพิมพ์เอง

แก้ `MY_PROMPT` เป็นอะไรก็ได้แล้วกดรัน ได้ภาพใหม่ทุกครั้งเพราะ seed สุ่ม

เขียนเป็นภาษาอังกฤษ เพราะ SDXL อ่านภาษาไทยไม่ออก
(ในเว็บจริงจะมีตัวแปลงจินตนาการภาษาไทยเป็นอังกฤษให้อัตโนมัติ)


In [ ]:
MY_PROMPT = 'isaanmudmee, intertwined naga serpents, emerald green ground, tiny stars scattered above'

import random
from IPython.display import display

seed = random.randrange(2**31)
generator = torch.Generator('cuda').manual_seed(seed)
image = pipe(
    prompt=MY_PROMPT, negative_prompt=NEGATIVE,
    num_inference_steps=28, guidance_scale=6.0,
    width=1024, height=1024, generator=generator,
).images[0]
print('seed', seed)
display(image.resize((640, 640)))


## 7. ดาวน์โหลด


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/preview', 'zip', '/content/preview')
files.download('/content/preview.zip')


---

## อ่านผลยังไง

ภาพที่ได้ตอนนี้คือ **เส้นฐาน** ยังไม่มีอะไรจากผ้า 20 ผืนของเราเลย

| ถ้าเห็นแบบนี้ | แปลว่า |
|---|---|
| เป็นผ้าลายเรขาคณิตทั่วโลก ไม่ใช่มัดหมี่อีสาน | ปกติ นี่คือช่องว่างที่ LoRA ต้องเติม |
| สีไม่ตรงกับที่สั่ง | ปกติ LoRA จะช่วยเรื่องนี้ได้มาก |
| ภาพเบลอ มีกรอบ มีตัวหนังสือ | ต้องปรับ negative prompt ก่อนเทรน |
| เป็นรูปถ่ายคนใส่ผ้า ไม่ใช่ผืนผ้า | ต้องปรับ prompt ก่อนเทรน |

สองแถวล่างคือสิ่งที่ควรแก้**ก่อน**เทรน เพราะเทรนแล้วก็ยังติดปัญหาเดิม

เทียบกับผ้าจริงใน `public/silk/` แล้วบอกได้เลยว่าอยากให้ต่างออกไปทางไหน
